### 1. infinite llm + tool call

```mermaid
flowchart TD
    A[LLM call] --> B{is tool call needed}
    B -- NO --> C[END]
    B -- YES --> D[Tool call]
    D --> A

```

In [2]:
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
openai_client = OpenAI()

In [4]:
def get_ticket_prices(city: str) -> str:
    FLIGHT_PRICES = {
        "london": "$550",
        "paris": "$600",
        "new york": "$450",
        "tokyo": "$950",
        "berlin": "$500"
    }
    cleaned_city = city.lower().strip()
    
    print(f"\n[Tool Executing] Looking up flight prices for: '{cleaned_city}'")
    
    if cleaned_city in FLIGHT_PRICES:
        price = FLIGHT_PRICES[cleaned_city]
        return f"The ticket for {city.title()} is {price}."
    
    return f"Tickets for {city.title()} are currently not available."

In [5]:
def get_weather(city: str) -> str:
    WEATHER_DATA = {
        "london": "12°C, Rainy",
        "paris": "15°C, Partly Cloudy",
        "new york": "18°C, Sunny",
        "tokyo": "20°C, Clear",
        "berlin": "10°C, Foggy"
    }
    cleaned_city = city.lower().strip()
    
    print(f"\n[Tool Executing] Looking up weather for: '{cleaned_city}'")
    
    if cleaned_city in WEATHER_DATA:
        condition = WEATHER_DATA[cleaned_city]
        return f"The weather in {city.title()} is currently {condition}."
    
    return f"Weather data for {city.title()} is currently not available."


In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_prices",
            "description": "Retrieves the flight ticket price for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to look up ticket prices for (e.g., 'Paris', 'London')."
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieves the current weather condition and temperature for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to look up the weather for (e.g., 'Paris', 'New York')."
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    }
]


In [7]:
TOOLS_DIRECTORY = {
    "get_ticket_prices": get_ticket_prices,
    "get_weather": get_weather
}

In [8]:
def handle_tool_calls(messages, tool_calls):
    for tool_call in tool_calls:
        try:
            tool_function = TOOLS_DIRECTORY.get(tool_call.function.name, None)
            tool_args = json.loads(tool_call.function.arguments)
        except Exception as e:
            # handle in better way
            raise e
        if tool_function:
            tool_result = tool_function(**tool_args)
            tool_message = {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_call.function.name,
                            "content": str(tool_result)
                        }
            messages.append(tool_message)
    return messages

In [9]:
def chat(user_prompt):
    system_message = {
        "role": "system",
        "content": "You are helpful airline assistant"
    }
    user_message = {
        "role": "user",
        "content": user_prompt
    }
    messages = [system_message] + [user_message]

    while True:
        print("---Calling LLM---")
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls

        if tool_calls:
            print("---Calling Tools---")
            messages = handle_tool_calls(messages, tool_calls)
            print("---Looping back to LLM call---")
            continue
        else:
            print("---Final response---")
            return response.choices[0].message.content

In [10]:
chat("Can you give me ticket price for london")

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'
---Looping back to LLM call---
---Calling LLM---
---Final response---


'The ticket price for London is $550.'

In [11]:
print(chat("Can you give me ticket price for london, berlin, munich, stuttgart, paris, san mateo, palo alto"))

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'

[Tool Executing] Looking up flight prices for: 'berlin'

[Tool Executing] Looking up flight prices for: 'munich'

[Tool Executing] Looking up flight prices for: 'stuttgart'

[Tool Executing] Looking up flight prices for: 'paris'

[Tool Executing] Looking up flight prices for: 'san mateo'

[Tool Executing] Looking up flight prices for: 'palo alto'
---Looping back to LLM call---
---Calling LLM---
---Final response---
Here are the ticket prices for the requested cities:

- **London**: $550
- **Berlin**: $500
- **Munich**: Tickets currently not available
- **Stuttgart**: Tickets currently not available
- **Paris**: $600
- **San Mateo**: Tickets currently not available
- **Palo Alto**: Tickets currently not available

If you need further assistance or information, feel free to ask!


In [12]:
print(chat("what is ticket price for london? "
    "if its ticket price is above $500 then check ticket price for new york and let me know."
    "if the ticket price for new york is below 500 then check the weather for new york."
     "if weather of new york is sunny then check weather for berlin"))

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'new york'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up weather for: 'new york'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up weather for: 'berlin'
---Looping back to LLM call---
---Calling LLM---
---Final response---
- The ticket price for London is $550.
- The ticket price for New York is $450.
- The weather in New York is sunny at 18°C.
- The weather in Berlin is foggy at 10°C.


### limited or capped llm + tool call

```mermaid
flowchart TD
    Init["i = 1\nmax_iters = 3"] --> LLM[LLM call]
    LLM --> Decision1{is tool\ncall needed?}
    
    Decision1 -- NO --> END1[END]
    Decision1 -- YES --> Decision2{is\ni <= max_iters?}
    
    Decision2 -- NO --> FinalLLM[Final LLM call] --> END2[END]
    Decision2 -- YES --> Tool[Tool call]
    
    Tool -- "i = i + 1" --> LLM


```

In [22]:
max_iterations = 3
max_tool_calls_at_one_time = 5

In [15]:
def handle_tool_calls(messages, tool_calls, max_tool_calls_at_one_time):
    for tool_call in tool_calls[:max_tool_calls_at_one_time]:
        try:
            tool_function = TOOLS_DIRECTORY.get(tool_call.function.name, None)
            tool_args = json.loads(tool_call.function.arguments)
        except Exception as e:
            # handle in better way
            raise e
        if tool_function:
            tool_result = tool_function(**tool_args)
            tool_message = {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_call.function.name,
                            "content": str(tool_result)
                        }
            messages.append(tool_message)
    return messages

In [16]:
def chat(user_prompt):
    system_message = {
        "role": "system",
        "content": "You are helpful airline assistant"
    }
    user_message = {
        "role": "user",
        "content": user_prompt
    }
    messages = [system_message] + [user_message]

    iteration = 1
    while iteration <= max_iterations:
        print("---Calling LLM---")
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls

        if tool_calls:
            print("---Calling Tools---")
            messages = handle_tool_calls(messages, tool_calls, max_tool_calls_at_one_time)
            iteration = iteration + 1
            print("---Looping back to LLM call---")
            continue
            
        else:
            print("---Final response---")
            return response.choices[0].message.content
    if messages[-1].get("role") == "tool":
        final_message = {
            "role": "system",
            "content": f"Max iterations have reached for LLM + Tool call {max_iterations}"
            "Please reply with whatever data available, Don't look for tool call"
            "Also please don't leak the data to the user that max iterations have reached"
        }
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return response.choices[0].message.content
    return "No Response"

In [17]:
print(chat("Can you give me ticket price for london, "))

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'
---Looping back to LLM call---
---Calling LLM---
---Final response---
The ticket price for London is $550.


In [18]:
print(chat("what is ticket price for london? "
    "if its ticket price is above $500 then check ticket price for new york and let me know."))

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'new york'
---Looping back to LLM call---
---Calling LLM---
---Final response---
The ticket price for London is $550, which is above $500. The ticket price for New York is $450.


In [19]:
print(chat("what is ticket price for london? "
    "if its ticket price is above $500 then check ticket price for new york and let me know."
    "if the ticket price for new york is below 500 then check the weather for new york."
    "if weather of new york is sunny then check weather for berlin"
    "if weather in berlin is rainy then check its ticket price"))

---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'london'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up flight prices for: 'new york'
---Looping back to LLM call---
---Calling LLM---
---Calling Tools---

[Tool Executing] Looking up weather for: 'new york'
---Looping back to LLM call---
The ticket price for London is $550, which is above $500. The ticket price for New York is $450, which is below $500. The weather in New York is sunny. 

Now, I will check the weather for Berlin. 
